In [0]:
cust_raw_df = spark.read.csv("/Volumes/wh_gb_test/gb_test/my_volume/cust_data/customer_data.csv", header=True, sep=",", inferSchema=True)
cust_raw_df.write.mode("overwrite").saveAsTable("wh_gb_test.wh_assignments.customer_data")

trans_raw_df = spark.read.csv("/Volumes/wh_gb_test/gb_test/my_volume/cust_data/sales_data.csv", header=True, sep=",", inferSchema=True)
trans_raw_df.write.mode("overwrite").saveAsTable("wh_gb_test.wh_assignments.transaction")

In [0]:
%sql
create table wh_gb_test.wh_assignments.customer_trans_detail as 
select CD.customer_id,gender,age,payment_method,td.invoice_no,category,quantity,price,invoice_date,shopping_mall from wh_gb_test.wh_assignments.customer_data as CD
inner join wh_gb_test.wh_assignments.transaction as TD on CD.customer_id = TD.customer_id;

In [0]:
%sql

select * from wh_gb_test.wh_assignments.transaction cd where not exists (select 'X' from wh_gb_test.wh_assignments.customer_trans_detail TD where CD.customer_id = TD.customer_id)

In [0]:
%sql
select date_format(cd.invoice_date, 'MMM') || ' ' || year(cd.invoice_date) as Month_YEAR, sum(cd.quantity*cd.price) as TOTAL_Revenue 
from wh_gb_test.wh_assignments.customer_trans_detail cd 
where cd.category='Clothing'
group by Month_YEAR

In [0]:
%sql
--usecase 3
--Who is the best customer? Find the customer who has spent the most money overall. Include their customer_id and name (from the customer table). Show the top 

select customer_id,cd.,sum(cd.quantity*cd.price) as money_spent from wh_gb_test.wh_assignments.customer_trans_detail cd 
group by customer_id
order by money_spent desc
limit 5


In [0]:
%sql
    
--For each customer, show their total revenue and classify them as 'High Value' (≥ 5000), 'Mid Value' (≥ 1000), or 'Low Value' (< 1000)
select customer_id,
sum(cd.quantity*cd.price) as money_spent,
case when sum(cd.quantity*cd.price) >=5000 then 'High Value'
when sum(cd.quantity*cd.price) >=1000 then 'Mid Value'
else 'Low Value'
end as customer_value
from wh_gb_test.wh_assignments.customer_trans_detail cd 
group by customer_id

In [0]:
%sql
select cd.gender,avg(cd.quantity*cd.price) as avg_transaction_value,sum(cd.quantity*cd.price) as total_revenue,count(*) as tranaction_count from wh_gb_test.wh_assignments.customer_trans_detail cd
group by cd.gender

In [0]:
%sql
select category, payment_method, count(*) as transaction_count
from wh_gb_test.wh_assignments.customer_trans_detail
group by category, payment_method
order by category, transaction_count desc

In [0]:
%sql
--The marketing team wants to target ads by age group. Bucket customers into: Teens (< 20), Young Adults (20–35), Adults (36–50), Seniors (50+). Which segment generates the most revenue?
select sum(cd.quantity*cd.price) as revenue,
case when cd.age < 20 then 'Teens'
when cd.age between 20 and 35 then 'Young Adults'
when cd.age between 36 and 50 then 'Adults'
when cd.age > 50 then 'Seniors' 
else 'Unknown' end as AGE_SEGMENTS from wh_gb_test.wh_assignments.customer_trans_detail cd 
group by AGE_SEGMENTS



In [0]:
%sql
--The retention team wants to run a win-back campaign. Identify all customers whose most recent purchase is more than 90 days before the latest invoice date in the dataset.
--consider max_date from the table as current_date

select customer_id,max(invoice_date) as last_purchase_date,datediff(current_date,last_purchase_date) as days_since_last from wh_gb_test.wh_assignments.customer_trans_detail
group by customer_id
having days_since_last > 90;



In [0]:
%sql
--Create a clean, analysis-ready master dataset by joining customer and transaction tables. Derive revenue, month, day-of-week, and age group. Handle nulls. Persist as a Parquet table.
create or replace table wh_gb_test.wh_assignments.Customer_golden_data as 
select cd.customer_id,
nvl(cd.gender,'Third Gender') as gender,
nvl(cd.age,0) as age,
cd.payment_method,
nvl(cd.shopping_mall,'Unknown') as shopping_mall,
nvl(cd.category,'Unknown') as category,
cd.invoice_date,
nvl(cd.price,0) as price,
nvl(cd.quantity,0) as quantity,
date_format(cd.invoice_date, 'MMM') || ' ' || year(cd.invoice_date) as Month_AND_YEAR,
date_format(cd.invoice_date, 'EEEE') as day_of_week,
cd.quantity*cd.price as revenue,
case when cd.age < 20 then 'Teens'
when cd.age between 20 and 35 then 'Young Adults'
when cd.age between 36 and 50 then 'Adults'
when cd.age > 50 then 'Seniors' 
else 'Unknown' end as AGE_SEGMENTS
from wh_gb_test.wh_assignments.customer_trans_detail cd

In [0]:
df = spark.table("wh_gb_test.wh_assignments.customer_golden_data")
df.write.mode("overwrite").parquet("/Volumes/wh_gb_test/gb_test/my_volume/cust_data/customer_golden_data.parquet")

In [0]:
%sql
select *  from wh_gb_test.wh_assignments.customer_trans_detail cd

In [0]:
%sql
-- usecase 11: Gender distribution across different product categories
select category, gender, count(*) as customer_count
from wh_gb_test.wh_assignments.customer_trans_detail
group by category, gender;

-- usecase 12: Total revenue generated in the year 2022
select sum(quantity * price) as total_revenue_2022
from wh_gb_test.wh_assignments.customer_trans_detail
where year(invoice_date) = 2022;